Import Libraries & Load Dataset

In [1]:
import pandas as pd
import plotly.express as px

from dash import Dash, html, dcc, Input, Output

# Load Dataset
df = pd.read_csv(r"C:\Users\hg\Downloads\Dataset_insuficiencia_renal_crnica.csv")

# Rename Spanish Columns to English
df.rename(columns={
    "Identificación del paciente": "Patient_ID",
    "Edad": "Age",
    "Presión arterial": "Blood_Pressure",
    "Gravedad específica de la orina": "Specific_Gravity",
    "Albúmina en orina": "Albumin",
    "Azúcar en orina": "Sugar",
    "Glóbulos rojos": "Red_Blood_Cells",
    "Células de pus": "Pus_Cells",
    "Grupos de células de pus": "Pus_Cell_Clumps",
    "Bacterias": "Bacteria",
    "Glucosa en sangre aleatoria": "Blood_Glucose_Random",
    "Urea en sangre": "Blood_Urea",
    "Creatinina sérica": "Serum_Creatinine",
    "Sodio en sangre": "Sodium",
    "Potasio en sangre": "Potassium",
    "Hemoglobina": "Hemoglobin",
    "Volumen corpuscular empaquetado": "Packed_Cell_Volume",
    "Recuento de glóbulos blancos": "White_Blood_Cell_Count",
    "Recuento de glóbulos rojos": "Red_Blood_Cell_Count",
    "Apetito": "Appetite",
    "Insuficiencia renal crónica": "Chronic_Kidney_Disease"
}, inplace=True)

print("Dataset Loaded Successfully!")
print("Dataset Shape:", df.shape)
print(df.head())

Dataset Loaded Successfully!
Dataset Shape: (15051, 21)
   Patient_ID  Age  Blood_Pressure  Specific_Gravity  Albumin  Sugar  \
0           0   48             178            1010.0      2.0      2   
1           1    7             138            1014.0      1.0      4   
2           2   62             165            1014.0      2.0      5   
3           3   48             141            1007.0      4.0      4   
4           4   51             175            1015.0      4.0      5   

  Red_Blood_Cells Pus_Cells Pus_Cell_Clumps  Bacteria  ...  Blood_Urea  \
0         anormal   anormal        presente  presente  ...       195.0   
1         anormal   anormal        presente  presente  ...       111.0   
2         anormal   anormal        presente  presente  ...       149.0   
3         anormal   anormal        presente  presente  ...        98.0   
4         anormal   anormal        presente  presente  ...       152.0   

   Serum_Creatinine  Sodium  Potassium  Hemoglobin  Packed_Cell_Vo

C:\Users\hg\AppData\Local\Temp\ipykernel_19256\2583293503.py:10: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={


Data Preparation

In [2]:
# Target Column
target = "Chronic_Kidney_Disease"

if target not in df.columns:
    raise ValueError("Target column was not found!")

# Standardize target values
df[target] = (
    df[target]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("í", "i", regex=False)
)

# Convert target values
target_mapping = {
    "si": "CKD",
    "yes": "CKD",
    "1": "CKD",
    "ckd": "CKD",
    "no": "No CKD",
    "0": "No CKD",
    "notckd": "No CKD"
}

df[target] = df[target].map(target_mapping)

# Remove rows with invalid target
df = df.dropna(subset=[target]).copy()

# Numerical Features
numeric_columns = [
    "Age",
    "Blood_Pressure",
    "Specific_Gravity",
    "Albumin",
    "Blood_Glucose_Random",
    "Blood_Urea",
    "Serum_Creatinine",
    "Sodium",
    "Potassium",
    "Hemoglobin",
    "Packed_Cell_Volume",
    "White_Blood_Cell_Count",
    "Red_Blood_Cell_Count"
]

# Convert numerical columns
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Available Numerical Features
numeric_features = [
    col for col in numeric_columns
    if col in df.columns and df[col].notna().any()
]

# Categorical Features
categorical_features = [
    col for col in [
        "Red_Blood_Cells",
        "Pus_Cells",
        "Pus_Cell_Clumps",
        "Bacteria",
        "Appetite",
        "Sugar"
    ]
    if col in df.columns
]

# Handle missing categorical values
for col in categorical_features:
    df[col] = df[col].fillna("Unknown").astype(str)

# Age Range
if df["Age"].notna().any():
    age_min = int(df["Age"].min())
    age_max = int(df["Age"].max())
else:
    age_min, age_max = 0, 100

if age_min == age_max:
    age_max += 1

if not numeric_features:
    raise ValueError("No numerical features found!")

if not categorical_features:
    raise ValueError("No categorical features found!")

print("Data Preparation Completed!")
print("Numerical Features:", numeric_features)
print("Categorical Features:", categorical_features)
print("Target Distribution:")
print(df[target].value_counts())

Data Preparation Completed!
Numerical Features: ['Age', 'Blood_Pressure', 'Specific_Gravity', 'Albumin', 'Blood_Glucose_Random', 'Blood_Urea', 'Serum_Creatinine', 'Sodium', 'Potassium', 'Hemoglobin', 'Packed_Cell_Volume', 'White_Blood_Cell_Count', 'Red_Blood_Cell_Count']
Categorical Features: ['Red_Blood_Cells', 'Pus_Cells', 'Pus_Cell_Clumps', 'Bacteria', 'Appetite', 'Sugar']
Target Distribution:
Chronic_Kidney_Disease
No CKD    7696
CKD       7353
Name: count, dtype: int64


Initialize Dash & Dashboard Layout

In [3]:
# Initialize Dash App
app = Dash(__name__)

app.title = "CKD Dashboard"

# Dashboard Layout
app.layout = html.Div(
    style={
        "fontFamily": "Arial",
        "backgroundColor": "#f4f6f9",
        "padding": "25px"
    },

    children=[

        # Header
        html.Div([
            html.H1(
                "Chronic Kidney Disease Dashboard",
                style={
                    "textAlign": "center",
                    "color": "#183153"
                }
            ),

            html.P(
                "Interactive Analysis of Kidney Disease Data",
                style={
                    "textAlign": "center",
                    "color": "#666",
                    "fontSize": "17px"
                }
            )
        ]),

        html.Hr(),

        # Filters
        html.Div([

            html.Div([
                html.Label("CKD Status"),

                dcc.Dropdown(
                    id="status-filter",
                    options=[
                        {"label": "All Patients", "value": "All"},
                        {"label": "CKD", "value": "CKD"},
                        {"label": "No CKD", "value": "No CKD"}
                    ],
                    value="All",
                    clearable=False
                )

            ], style={
                "width": "32%",
                "display": "inline-block"
            }),

            html.Div([
                html.Label("Age Range"),

                dcc.RangeSlider(
                    id="age-filter",
                    min=age_min,
                    max=age_max,
                    value=[age_min, age_max],
                    marks={
                        age_min: str(age_min),
                        age_max: str(age_max)
                    },
                    step=1,
                    tooltip={
                        "placement": "bottom",
                        "always_visible": True
                    }
                )

            ], style={
                "width": "60%",
                "display": "inline-block",
                "marginLeft": "5%"
            })

        ], style={
            "backgroundColor": "white",
            "padding": "20px",
            "borderRadius": "10px",
            "marginBottom": "20px"
        }),

        # KPI Cards
        html.Div([

            html.Div([
                html.H4("Total Patients"),
                html.H2(id="total-patients")
            ], className="kpi-card"),

            html.Div([
                html.H4("CKD Patients"),
                html.H2(id="ckd-patients")
            ], className="kpi-card"),

            html.Div([
                html.H4("No CKD Patients"),
                html.H2(id="no-ckd-patients")
            ], className="kpi-card"),

            html.Div([
                html.H4("CKD Percentage"),
                html.H2(id="ckd-percentage")
            ], className="kpi-card")

        ], style={
            "display": "grid",
            "gridTemplateColumns": "repeat(4, 1fr)",
            "gap": "15px",
            "marginBottom": "20px"
        }),

        # Pie Chart and Age Histogram
        html.Div([

            html.Div([
                dcc.Graph(id="ckd-pie-chart")
            ], style={
                "width": "49%",
                "backgroundColor": "white",
                "padding": "10px",
                "borderRadius": "10px"
            }),

            html.Div([
                dcc.Graph(id="age-histogram")
            ], style={
                "width": "49%",
                "backgroundColor": "white",
                "padding": "10px",
                "borderRadius": "10px"
            })

        ], style={
            "display": "flex",
            "justifyContent": "space-between",
            "marginBottom": "20px"
        }),

        # Numerical Feature Analysis
        html.Div([

            html.H3("Numerical Feature Analysis"),

            html.P(
                "Compare average medical measurements "
                "between CKD and No CKD patients.",
                style={"color": "#666"}
            ),

            dcc.Dropdown(
                id="numeric-feature",

                options=[
                    {
                        "label": col.replace("_", " "),
                        "value": col
                    }
                    for col in numeric_features
                ],

                value=(
                    "Serum_Creatinine"
                    if "Serum_Creatinine" in numeric_features
                    else numeric_features[0]
                ),

                clearable=False
            ),

            dcc.Graph(id="numeric-bar-chart")

        ], style={
            "backgroundColor": "white",
            "padding": "15px",
            "borderRadius": "10px",
            "marginBottom": "20px"
        }),

        # Categorical Feature Analysis
        html.Div([

            html.H3("Categorical Feature Analysis"),

            dcc.Dropdown(
                id="categorical-feature",

                options=[
                    {
                        "label": col.replace("_", " "),
                        "value": col
                    }
                    for col in categorical_features
                ],

                value=(
                    "Red_Blood_Cells"
                    if "Red_Blood_Cells" in categorical_features
                    else categorical_features[0]
                ),

                clearable=False
            ),

            dcc.Graph(id="categorical-bar-chart")

        ], style={
            "backgroundColor": "white",
            "padding": "15px",
            "borderRadius": "10px"
        }),

        # Footer
        html.P(
            "CKD Data Analysis Dashboard",
            style={
                "textAlign": "center",
                "color": "#888",
                "marginTop": "25px"
            }
        )

    ]
)

# Dashboard Styling
app.index_string = """
<!DOCTYPE html>
<html>
    <head>
        {%metas%}
        <title>CKD Dashboard</title>
        {%favicon%}
        {%css%}

        <style>
            .kpi-card {
                background: white;
                padding: 15px;
                border-radius: 10px;
                text-align: center;
                box-shadow: 0 2px 6px rgba(0,0,0,0.08);
                color: #183153;
            }

            .kpi-card h2 {
                color: #1677b8;
                font-size: 30px;
            }

            body {
                margin: 0;
            }
        </style>
    </head>

    <body>
        {%app_entry%}
        <footer>
            {%config%}
            {%scripts%}
            {%renderer%}
        </footer>
    </body>
</html>
"""

print("Dashboard Layout Created Successfully!")

Dashboard Layout Created Successfully!


Dashboard Callbacks

In [4]:
@app.callback(

    Output("total-patients", "children"),
    Output("ckd-patients", "children"),
    Output("no-ckd-patients", "children"),
    Output("ckd-percentage", "children"),

    Output("ckd-pie-chart", "figure"),
    Output("age-histogram", "figure"),
    Output("numeric-bar-chart", "figure"),
    Output("categorical-bar-chart", "figure"),

    Input("status-filter", "value"),
    Input("age-filter", "value"),
    Input("numeric-feature", "value"),
    Input("categorical-feature", "value")

)

def update_dashboard(status, age_range, numeric_feature, categorical_feature):

    # Apply Filters
    filtered_df = df.copy()

    if status != "All":
        filtered_df = filtered_df[
            filtered_df[target] == status
        ]

    filtered_df = filtered_df[
        filtered_df["Age"].between(
            age_range[0],
            age_range[1]
        )
    ]

    # KPI Calculations
    total = len(filtered_df)

    ckd_count = (
        filtered_df[target] == "CKD"
    ).sum()

    no_ckd_count = (
        filtered_df[target] == "No CKD"
    ).sum()

    percentage = (
        (ckd_count / total) * 100
        if total > 0 else 0
    )

    # Handle Empty Data
    if total == 0:

        empty_fig = px.scatter(
            title="No data available for selected filters"
        )

        return (
            "0",
            "0",
            "0",
            "0%",
            empty_fig,
            empty_fig,
            empty_fig,
            empty_fig
        )

    # ======================================
    # 1. CKD Distribution Pie Chart
    # ======================================

    status_counts = (
        filtered_df[target]
        .value_counts()
        .reindex(["CKD", "No CKD"], fill_value=0)
        .reset_index()
    )

    status_counts.columns = ["Status", "Count"]

    pie_fig = px.pie(
        status_counts,
        names="Status",
        values="Count",
        title="CKD Distribution",
        hole=0.4,
        color="Status",

        color_discrete_map={
            "CKD": "#e74c3c",
            "No CKD": "#3498db"
        }
    )

    # ======================================
    # 2. Age Distribution Histogram
    # ======================================

    age_fig = px.histogram(
        filtered_df,
        x="Age",
        color=target,
        title="Age Distribution by CKD Status",
        barmode="overlay",
        opacity=0.75,

        color_discrete_map={
            "CKD": "#e74c3c",
            "No CKD": "#3498db"
        }
    )

    # ======================================
    # 3. Numerical Feature Bar Chart
    # ======================================

    numeric_data = (
        filtered_df
        .groupby(target)[numeric_feature]
        .mean()
        .reindex(["CKD", "No CKD"])
        .dropna()
        .reset_index()
    )

    numeric_fig = px.bar(
        numeric_data,
        x=target,
        y=numeric_feature,
        color=target,

        title=(
            f"Average {numeric_feature.replace('_', ' ')} "
            "by CKD Status"
        ),

        color_discrete_map={
            "CKD": "#e74c3c",
            "No CKD": "#3498db"
        },

        labels={
            target: "Patient Status",
            numeric_feature: "Average Value"
        }
    )

    numeric_fig.update_traces(
        texttemplate="%{y:.2f}",
        textposition="outside"
    )

    numeric_fig.update_layout(
        template="plotly_white",
        showlegend=False,
        yaxis_title="Average Value",
        xaxis_title="Patient Status"
    )

    # ======================================
    # 4. Categorical Feature Bar Chart
    # ======================================

    category_data = (
        filtered_df
        .groupby([categorical_feature, target])
        .size()
        .reset_index(name="Count")
    )

    category_fig = px.bar(
        category_data,
        x=categorical_feature,
        y="Count",
        color=target,
        barmode="group",

        title=(
            f"{categorical_feature.replace('_', ' ')} "
            "vs CKD Status"
        ),

        color_discrete_map={
            "CKD": "#e74c3c",
            "No CKD": "#3498db"
        }
    )

    category_fig.update_layout(
        template="plotly_white",
        xaxis_title=categorical_feature.replace("_", " "),
        yaxis_title="Number of Patients"
    )

    # ======================================
    # Return Dashboard Results
    # ======================================

    return (

        f"{total:,}",
        f"{ckd_count:,}",
        f"{no_ckd_count:,}",
        f"{percentage:.1f}%",

        pie_fig,
        age_fig,
        numeric_fig,
        category_fig

    )


print("Dashboard Callbacks Created Successfully!")

Dashboard Callbacks Created Successfully!


Run Dashboard

In [5]:
app.run(debug=True, port=8050)